In [81]:
from pathlib import Path
from functools import reduce

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()

base_path = Path(r"C:\Users\selim\Desktop\Motosport_Data_Platform\data_lake\gold\fastf1")

gold_tables = [
    "gold_constructor_race_summary",
    "gold_constructor_recent_form",
    "gold_driver_race_summary",
    "gold_driver_recent_form",
    "gold_lap_prediction_dataset",
]

def read_all_seasons_for_table(table_name: str):
    dfs = []

    print(f"\n===== Loading {table_name} =====")

    for season_dir in sorted(base_path.iterdir()):
        if not season_dir.is_dir():
            continue

        table_path = season_dir / table_name

        if not table_path.exists():
            print(f"Skipping {season_dir.name}: table not found")
            continue

        print(f"Loading {season_dir.name}")
        df = spark.read.parquet(str(table_path))
        dfs.append(df)

    if not dfs:
        raise ValueError(f"No data found for table: {table_name}")

    return reduce(
        lambda a, b: a.unionByName(b, allowMissingColumns=True),
        dfs
    )

gold_constructor_race_summary_df = read_all_seasons_for_table("gold_constructor_race_summary")
gold_constructor_recent_form_df = read_all_seasons_for_table("gold_constructor_recent_form")
gold_driver_race_summary_df = read_all_seasons_for_table("gold_driver_race_summary")
gold_driver_recent_form_df = read_all_seasons_for_table("gold_driver_recent_form")
gold_lap_prediction_dataset_df = read_all_seasons_for_table("gold_lap_prediction_dataset")



===== Loading gold_constructor_race_summary =====
Loading season_2018
Loading season_2019
Loading season_2020
Loading season_2021
Loading season_2022
Loading season_2023
Loading season_2024
Loading season_2025

===== Loading gold_constructor_recent_form =====
Loading season_2018
Loading season_2019
Loading season_2020
Loading season_2021
Loading season_2022
Loading season_2023
Loading season_2024
Loading season_2025

===== Loading gold_driver_race_summary =====
Loading season_2018
Loading season_2019
Loading season_2020
Loading season_2021
Loading season_2022
Loading season_2023
Loading season_2024
Loading season_2025

===== Loading gold_driver_recent_form =====
Loading season_2018
Loading season_2019
Loading season_2020
Loading season_2021
Loading season_2022
Loading season_2023
Loading season_2024
Loading season_2025

===== Loading gold_lap_prediction_dataset =====
Loading season_2018
Loading season_2019
Loading season_2020
Loading season_2021
Loading season_2022
Loading season_2023

In [82]:
print("\n===== FINAL COUNTS =====")
print("gold_constructor_race_summary:", gold_constructor_race_summary_df.count())
print("gold_constructor_recent_form:", gold_constructor_recent_form_df.count())
print("gold_driver_race_summary:", gold_driver_race_summary_df.count())
print("gold_driver_recent_form:", gold_driver_recent_form_df.count())
print("gold_lap_prediction_dataset:", gold_lap_prediction_dataset_df.count())

print("\n===== HEADS =====")

print("\n gold_constructor_race_summary_df:")
gold_constructor_race_summary_df.show(5, truncate=False)

print("\n gold_constructor_recent_form_df:")
gold_constructor_recent_form_df.show(5, truncate=False)

print("\n gold_driver_race_summary_df:")
gold_driver_race_summary_df.show(5, truncate=False)

print("\n gold_driver_recent_form_df:")
gold_driver_recent_form_df.show(5, truncate=False)

print("\n gold_lap_prediction_dataset_df:")
gold_lap_prediction_dataset_df.show(5, truncate=False)


===== FINAL COUNTS =====
gold_constructor_race_summary: 1529
gold_constructor_recent_form: 1529
gold_driver_race_summary: 2981
gold_driver_recent_form: 2981
gold_lap_prediction_dataset: 164699

===== HEADS =====

 gold_constructor_race_summary_df:
+------+-------------+---------------+-----------------------+-----------+-------------------+--------------------+-------------------+------------------+----------------+---------------+---------------+---------------+---------+
|season|race         |team_name      |race_date              |num_drivers|avg_finish_position|best_finish_position|total_position_gain|avg_lap_time_ms   |best_lap_time_ms|total_pit_stops|top_10_finishes|podium_finishes|dnf_count|
+------+-------------+---------------+-----------------------+-----------+-------------------+--------------------+-------------------+------------------+----------------+---------------+---------------+---------------+---------+
|2018  |united_states|McLaren        |2018-10-21 18:13:38.071

In [83]:
driver_cols_to_rename = [
    "avg_finish_last_3",
    "avg_finish_last_5",
    "avg_position_gain_last_5",
    "top_10_count_last_5",
    "podium_count_last_5",
    "dnf_count_last_5",
]

for col_name in driver_cols_to_rename:
    gold_driver_recent_form_df = gold_driver_recent_form_df.withColumnRenamed(
        col_name,
        f"driver_{col_name}"
    )

In [84]:
constructor_cols_to_rename = [
    "avg_finish_last_3",
    "avg_finish_last_5",
    "avg_position_gain_last_5",
    "top_10_count_last_5",
    "podium_count_last_5",
    "dnf_count_last_5",
]

for col_name in constructor_cols_to_rename:
    gold_constructor_recent_form_df = gold_constructor_recent_form_df.withColumnRenamed(
        col_name,
        f"constructor_{col_name}"
    )

In [85]:
print("\n gold_constructor_recent_form_df:")
gold_constructor_recent_form_df.show(5, truncate=False)
print("\n gold_driver_recent_form_df:")
gold_driver_recent_form_df.show(5, truncate=False)


 gold_constructor_recent_form_df:
+------+----------+---------+-----------------------+-----------+-------------------+--------------------+-------------------+------------------+----------------+---------------+---------------+---------------+---------+-----------------------------+-----------------------------+------------------------------------+-------------------------------+-------------------------------+----------------------------+
|season|race      |team_name|race_date              |num_drivers|avg_finish_position|best_finish_position|total_position_gain|avg_lap_time_ms   |best_lap_time_ms|total_pit_stops|top_10_finishes|podium_finishes|dnf_count|constructor_avg_finish_last_3|constructor_avg_finish_last_5|constructor_avg_position_gain_last_5|constructor_top_10_count_last_5|constructor_podium_count_last_5|constructor_dnf_count_last_5|
+------+----------+---------+-----------------------+-----------+-------------------+--------------------+-------------------+-----------------

In [86]:

driver_recent_features = [
    "season",
    "race",
    "driver",
    "driver_avg_finish_last_3",
    "driver_avg_finish_last_5",
    "driver_avg_position_gain_last_5",
    "driver_top_10_count_last_5",
    "driver_podium_count_last_5",
    "driver_dnf_count_last_5",
]

constructor_recent_features = [
    "season",
    "race",
    "team_name",
    "constructor_avg_finish_last_3",
    "constructor_avg_finish_last_5",
    "constructor_avg_position_gain_last_5",
    "constructor_top_10_count_last_5",
    "constructor_podium_count_last_5",
    "constructor_dnf_count_last_5",
]

driver_recent_df = gold_driver_recent_form_df.select(*driver_recent_features)

constructor_recent_df = gold_constructor_recent_form_df.select(*constructor_recent_features)

ml_feature_df = (
    gold_lap_prediction_dataset_df
    .join(
        driver_recent_df,
        on=["season", "race", "driver"],
        how="left"
    )
    .join(
        constructor_recent_df,
        on=["season", "race", "team_name"],
        how="left"
    )
)

print("Rows:", ml_feature_df.count())
print("Columns:", len(ml_feature_df.columns))

ml_feature_df.show(10, truncate=False)

Rows: 164699
Columns: 46
+------+---------+---------+------+-------------+----------+-----------------------+--------+-----+---------+---------+----------+----------+-------------+--------------+--------+---------+-------------+-----------+------+--------+--------+--------+----------+--------------+----------+----------+----------------+-----------------+------------------+----------------------+-------------------+-------------------+------------------+------------------------+------------------------+-------------------------------+--------------------------+--------------------------+-----------------------+-----------------------------+-----------------------------+------------------------------------+-------------------------------+-------------------------------+----------------------------+
|season|race     |team_name|driver|driver_number|lap_number|lapstartdate           |position|stint|compound |tyre_life|fresh_tyre|is_pit_lap|is_pit_in_lap|is_pit_out_lap|is_green|is_yellow|is

In [87]:
from pyspark.sql.functions import col, when
from pyspark.sql.window import Window
from pyspark.sql.functions import first

# Tyre life is 0 when null (happens only on race start)
ml_feature_df = ml_feature_df.withColumn(
    "tyre_life",
    when(col("tyre_life").isNull(), 0).otherwise(col("tyre_life"))
)

#change compound nan to next lap compound (happens only on lap 1)
ml_feature_df = ml_feature_df.withColumn(
    "compound",
    when(col("compound") == "nan", None).otherwise(col("compound"))
)

window_next = Window.partitionBy("season", "race", "driver") \
    .orderBy(col("lap_number").asc()) \
    .rowsBetween(0, Window.unboundedFollowing)

ml_feature_df = ml_feature_df.withColumn(
    "compound",
    first("compound", ignorenulls=True).over(window_next)
)
ml_feature_df = ml_feature_df.filter(col("compound").isNotNull())

# Stint is 1 when null (happens only on race start)
ml_feature_df = ml_feature_df.withColumn(
    "stint",
    when((col("stint").isNull()), 1)
    .otherwise(col("stint"))
) 
# Change boolean to 0 and 1

bool_cols = [
    "fresh_tyre", "is_pit_lap", "is_pit_in_lap", "is_pit_out_lap",
    "is_green", "is_yellow", "is_safety_car", "is_red_flag", "is_vsc",
    "last_lap_is_green", "last_lap_is_yellow", "last_lap_is_safety_car", 
    "is_raining"
]

for c in bool_cols:
    ml_feature_df = ml_feature_df.withColumn(c, col(c).cast("int"))


ml_feature_df.show(10, truncate=False)

+------+---------+---------+------+-------------+----------+-----------------------+--------+-----+---------+---------+----------+----------+-------------+--------------+--------+---------+-------------+-----------+------+--------+--------+--------+----------+--------------+----------+----------+----------------+-----------------+------------------+----------------------+-------------------+-------------------+------------------+------------------------+------------------------+-------------------------------+--------------------------+--------------------------+-----------------------+-----------------------------+-----------------------------+------------------------------------+-------------------------------+-------------------------------+----------------------------+
|season|race     |team_name|driver|driver_number|lap_number|lapstartdate           |position|stint|compound |tyre_life|fresh_tyre|is_pit_lap|is_pit_in_lap|is_pit_out_lap|is_green|is_yellow|is_safety_car|is_red_flag|i

In [88]:
from pyspark.sql.functions import col, sum as spark_sum, when

null_counts = ml_feature_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ml_feature_df.columns
])

null_counts.show(truncate=False)

+------+----+---------+------+-------------+----------+------------+--------+-----+--------+---------+----------+----------+-------------+--------------+--------+---------+-------------+-----------+------+--------+--------+--------+----------+--------------+----------+----------+----------------+-----------------+------------------+----------------------+-------------------+-------------------+------------------+------------------------+------------------------+-------------------------------+--------------------------+--------------------------+-----------------------+-----------------------------+-----------------------------+------------------------------------+-------------------------------+-------------------------------+----------------------------+
|season|race|team_name|driver|driver_number|lap_number|lapstartdate|position|stint|compound|tyre_life|fresh_tyre|is_pit_lap|is_pit_in_lap|is_pit_out_lap|is_green|is_yellow|is_safety_car|is_red_flag|is_vsc|air_temp|humidity|pressure|t

In [89]:
# Add a feature column to indicate if past values are missing for rolling averages 

ml_feature_df = ml_feature_df.withColumn(
    "driver_history_available",
    when(col("driver_avg_finish_last_3").isNull(), 0).otherwise(1)
)

ml_feature_df = ml_feature_df.withColumn(
    "constructor_history_available",
    when(col("constructor_avg_finish_last_3").isNull(), 0).otherwise(1)
)

ml_feature_df = ml_feature_df.withColumn(
    "last_lap_history_available",
    when(col("last_lap_time_ms").isNull(), 0).otherwise(1)
)

# Replace nulls in rolling average features with a default value
ml_feature_df = ml_feature_df.fillna({
    "driver_avg_finish_last_3": 10,
    "driver_avg_finish_last_5": 10,
    "driver_avg_position_gain_last_5": 0,
    "driver_top_10_count_last_5": 0,
    "driver_podium_count_last_5": 0,
    "driver_dnf_count_last_5": 0,

    "constructor_avg_finish_last_3": 10,
    "constructor_avg_finish_last_5": 10,
    "constructor_avg_position_gain_last_5": 0,
    "constructor_top_10_count_last_5": 0,
    "constructor_podium_count_last_5": 0,
    "constructor_dnf_count_last_5": 0,

    "last_lap_time_ms": 0,
    "last_lap_is_green": 0,
    "last_lap_is_yellow": 0,
    "last_lap_is_safety_car": 0,
    "avg_lap_time_last_2": 0,
    "avg_lap_time_last_3": 0,
})


In [90]:
null_counts = ml_feature_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ml_feature_df.columns
])

null_counts.show(truncate=False)

+------+----+---------+------+-------------+----------+------------+--------+-----+--------+---------+----------+----------+-------------+--------------+--------+---------+-------------+-----------+------+--------+--------+--------+----------+--------------+----------+----------+----------------+-----------------+------------------+----------------------+-------------------+-------------------+------------------+------------------------+------------------------+-------------------------------+--------------------------+--------------------------+-----------------------+-----------------------------+-----------------------------+------------------------------------+-------------------------------+-------------------------------+----------------------------+------------------------+-----------------------------+--------------------------+
|season|race|team_name|driver|driver_number|lap_number|lapstartdate|position|stint|compound|tyre_life|fresh_tyre|is_pit_lap|is_pit_in_lap|is_pit_out_la

In [91]:
# Sort the final dataset to have chronological order
ml_feature_df = ml_feature_df.orderBy(
    col("lapstartdate"),
    col("driver"),
    col("lap_number")
)
 
# Drop columns that won't be used in modeling
ml_feature_df = ml_feature_df.drop("lapstartdate", "driver_number")
ml_feature_df.show(10, truncate=False)

+------+---------+------------+------+----------+--------+-----+---------+---------+----------+----------+-------------+--------------+--------+---------+-------------+-----------+------+--------+--------+--------+----------+--------------+----------+----------+----------------+-----------------+------------------+----------------------+-------------------+-------------------+------------------+------------------------+------------------------+-------------------------------+--------------------------+--------------------------+-----------------------+-----------------------------+-----------------------------+------------------------------------+-------------------------------+-------------------------------+----------------------------+------------------------+-----------------------------+--------------------------+
|season|race     |team_name   |driver|lap_number|position|stint|compound |tyre_life|fresh_tyre|is_pit_lap|is_pit_in_lap|is_pit_out_lap|is_green|is_yellow|is_safety_car|i

In [92]:
import pandas as pd

ml_pd_feature_df = ml_feature_df.toPandas()

# One-hot encode categorical variables
categorical_cols = ["race", "team_name", "driver", "compound"]
ml_pd_feature_df = pd.get_dummies(ml_pd_feature_df, columns=categorical_cols)

In [93]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

ml_pd_feature_df.head()

,season,lap_number,position,stint,tyre_life,fresh_tyre,is_pit_lap,is_pit_in_lap,is_pit_out_lap,is_green,is_yellow,is_safety_car,is_red_flag,is_vsc,air_temp,humidity,pressure,track_temp,wind_direction,wind_speed,is_raining,last_lap_time_ms,last_lap_is_green,last_lap_is_yellow,last_lap_is_safety_car,avg_lap_time_last_2,avg_lap_time_last_3,target_lap_time_ms,driver_avg_finish_last_3,driver_avg_finish_last_5,driver_avg_position_gain_last_5,driver_top_10_count_last_5,driver_podium_count_last_5,driver_dnf_count_last_5,constructor_avg_finish_last_3,constructor_avg_finish_last_5,constructor_avg_position_gain_last_5,constructor_top_10_count_last_5,constructor_podium_count_last_5,constructor_dnf_count_last_5,driver_history_available,constructor_history_available,last_lap_history_available,race_abu_dhabi,race_australia,race_austria,race_azerbaijan,race_bahrain,race_belgium,race_brazil,race_canada,race_china,race_france,race_germany,race_great_britain,race_hungary,race_italy,race_japan,race_mexico,race_monaco,race_netherlands,race_portugal,race_qatar,race_russia,race_saudi_arabia,race_singapore,race_spain,race_turkey,race_united_arab_emirates,race_united_kingdom,race_united_states,team_name_Alfa Romeo,team_name_Alfa Romeo Racing,team_name_AlphaTauri,team_name_Alpine,team_name_Aston Martin,team_name_Ferrari,team_name_Force India,team_name_Haas F1 Team,team_name_Kick Sauber,team_name_McLaren,team_name_Mercedes,team_name_RB,team_name_Racing Bulls,team_name_Racing Point,team_name_Red Bull Racing,team_name_Renault,team_name_Sauber,team_name_Toro Rosso,team_name_Williams,driver_ALB,driver_ALO,driver_ANT,driver_BEA,driver_BOR,driver_BOT,driver_COL,driver_DEV,driver_DOO,driver_ERI,driver_FIT,driver_GAS,driver_GIO,driver_GRO,driver_HAD,driver_HAM,driver_HAR,driver_HUL,driver_KUB,driver_KVY,driver_LAT,driver_LAW,driver_LEC,driver_MAG,driver_MAZ,driver_MSC,driver_NOR,driver_OCO,driver_PER,driver_PIA,driver_RAI,driver_RIC,driver_RUS,driver_SAI,driver_SAR,driver_SIR,driver_STR,driver_TSU,driver_VAN,driver_VER,driver_VET,driver_ZHO,compound_HARD,compound_HYPERSOFT,compound_INTERMEDIATE,compound_MEDIUM,compound_None,compound_SOFT,compound_SUPERSOFT,compound_ULTRASOFT,compound_UNKNOWN,compound_WET
0,2018,1,10,1,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,101528.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
1,2018,1,15,1,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,103824.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
2,2018,1,16,1,0,1,0,0,0,0,0,0,0,0,24.2,36.3,997.0,38.9,307.0,4.1,0,0.0,0,0,0,0.0,0.0,104674.0,10.0,10.0,0.0,0,0,0,10.0,10.0,0.0,0,0,0,0,0,0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,